# NB02 - Techniques d'ingestion de données

Objéctifs : 
- Ingéstion de données dans Delta Lake avec `CREATE TABLE AS SELECT`
- Chargement incrémental avec `COPY INTO`
- Chargement en temps réel en utilisant les pipelines **Auto Loader**
- Lakeflow Connect

In [0]:
%run "./Resources/NB02/Setup"

**Différence entre `CREATE OR REPLACE TABLE` et `COPY INTO` :**

- `CREATE OR REPLACE TABLE` :  
  Crée une nouvelle table ou remplace une table existante avec le résultat d'une requête. Utilisé pour définir la structure et le contenu d'une table à partir de données existantes.

- `COPY INTO` :  
  Charge des données de façon incrémentale dans une table à partir de fichiers externes (par exemple, fichiers CSV, Parquet, etc.). Utilisé pour ajouter ou mettre à jour des données sans remplacer la table entière.

In [0]:
%sql
CREATE OR REPLACE TABLE crime_states_bronze
  USING DELTA AS
  SELECT * FROM parquet.`/Volumes/demo_{username}/demo_delta_lake/dld_demo/donnee-comm-data.gouv-parquet-2024-geographie2025-produit-le2025-06-04.parquet`;

DESCRIBE EXTENDED crime_states_bronze;

In [0]:
%sql
DROP table IF EXISTS crime_states_bronze;
create table crime_states_bronze USING DELTA;

In [0]:
%sql
COPY INTO crime_states_bronze
FROM '/Volumes/demo_{username}/demo_delta_lake/dld_demo/donnee-comm-data.gouv-parquet-2024-geographie2025-produit-le2025-06-04.parquet'
FILEFORMAT = PARQUET
COPY_OPTIONS ('mergeSchema' = 'true');



La requête `%sql COPY INTO crime_states_bronze ...` permet de charger de façon incrémentale des données depuis un fichier Parquet externe vers la table `crime_states_bronze`. 

- **FROM** : Spécifie le chemin du fichier source à importer.
- **FILEFORMAT = PARQUET** : Indique le format du fichier à charger.
- **COPY_OPTIONS ('mergeSchema' = 'true')** : Permet d'adapter automatiquement le schéma de la table si le fichier source contient de nouvelles colonnes.

Cette commande ajoute ou met à jour les données dans la table sans la remplacer entièrement.

In [0]:
%sql
select count(*) from crime_states_bronze;

In [0]:
%sql
COPY INTO crime_states_bronze
FROM '/Volumes/demo_{username}}/demo_delta_lake/dld_demo/donnee-comm-data.gouv-parquet-2024-geographie2025-produit-le2025-06-04.parquet'
FILEFORMAT = PARQUET
COPY_OPTIONS ('mergeSchema' = 'true');

select count(*) from crime_states_bronze;

La commande `COPY INTO` est **idempotente**, ce qui signifie que l'exécution répétée de la même commande ne duplique pas les données déjà chargées. Databricks garde une trace des fichiers importés et ne recharge que les nouveaux fichiers ou les fichiers modifiés, garantissant ainsi l'intégrité des données lors de multiples exécutions.

### Fonctions intégrées (built-in) de Databricks SQL

- [Documentation Databricks SQL Built-in Functions](https://docs.databricks.com/en/sql/language-manual/functions.html)

In [0]:
%sql
drop table if exists crime_states_bronze;
create table crime_states_bronze using delta;
    
COPY INTO crime_states_bronze
FROM (
  SELECT 
    current_timestamp() process_time,
    _metadata.file_name source_file,
    *
    FROM '/Volumes/demo_{username}/demo_delta_lake/dld_demo/donnee-comm-data.gouv-parquet-2024-geographie2025-produit-le2025-06-04.parquet'
)
FILEFORMAT = PARQUET
COPY_OPTIONS ('mergeSchema' = 'true');

select * from crime_states_bronze;

### Databricks Auto Loader

Auto Loader est une fonctionnalité de Databricks permettant de charger automatiquement et en continu des fichiers nouveaux ou modifiés depuis un dossier source (par exemple, un dossier cloud) vers une table Delta Lake. Il détecte les nouveaux fichiers sans avoir à recharger l'ensemble des données, ce qui facilite l'ingestion en temps réel ou quasi temps réel.

- **Avantages** :
  - Détection automatique des nouveaux fichiers.
  - Ingestion scalable et efficace.
  - Gestion du schéma évolutif.
  - Idéal pour les flux de données en continu.

- **Utilisation typique** :
  - Chargement de données brutes dans une table bronze.
  - Traitement de données en streaming avec Spark Structured Streaming.

> [Documentation officielle Auto Loader](https://docs.databricks.com/en/ingestion/auto-loader/index.html)

In [0]:
# Exemple d'utilisation de l'Auto Loader avec un volume Databricks
dbutils.fs.rm("/Volumes/demo_edadou/demo_delta_lake/dld_demo/checkpoint", recurse=True)

df = (spark.readStream
      .format("cloudFiles")
      .option("cloudFiles.format", "json")
      .option("cloudFiles.maxFilesPerTrigger", 1)
      .option("cloudFiles.inferSchema", "true")
      .option("cloudFiles.schemaLocation", "/Volumes/demo_edadou/demo_delta_lake/dld_demo/_schemas")
      .load("/Volumes/demo_edadou/demo_delta_lake/dld_demo")
     )

display(df, checkpointLocation="/Volumes/demo_edadou/demo_delta_lake/dld_demo/checkpoint")

### Lakeflow Connect

Lakeflow Connect est une fonctionnalité de Databricks permettant de connecter et d'ingérer facilement des données provenant de sources externes (bases de données, applications SaaS, fichiers, etc.) vers Delta Lake. Elle simplifie l'intégration de données en automatisant la connexion, l'extraction, la transformation et le chargement (ETL) dans le Lakehouse.

- **Avantages** :
  - Connexion rapide à de nombreuses sources de données.
  - Automatisation des flux d'ingestion.
  - Gestion des transformations et du mapping de schéma.
  - Surveillance et orchestration intégrées.

> [Documentation officielle Lakeflow Connect](https://docs.databricks.com/en/lakeflow/connect/index.html)